# 07 Blinded human review of interpolation trajectories

This notebook complements Notebook 05 with an exploratory human review of visual plausibility, visible progression toward Ankle Boot, and abrupt changes. It uses a preregistered random subset of the same paired test cohort and all five methods. Method codes are randomized separately within each pair and kept in a separate key file. The images are the evidence -> Classifier scores are not shown to raters.

Note: This is a pilot study, not a large-scale study designed to produce statistically representative results

In [ ]:
from pathlib import Path
import csv
import json
import re
import secrets
import sys
from zipfile import ZIP_DEFLATED, ZipFile

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
repo_candidates = [cwd, cwd / 'minRF_interpolation']
repo_candidates.extend(
    candidate
    for parent in cwd.parents
    for candidate in (parent, parent / 'minRF_interpolation')
)
repo_candidates.extend((Path('/content/minRF_interpolation'), Path('/content/minrf/minRF_interpolation')))
repo_root = next(
    (candidate for candidate in repo_candidates if (candidate / 'api' / 'flow_model_api.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Repository not found. Run notebooks/00_colab_setup.ipynb first.')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from notebooks.utils.evaluation_workflow import bootstrap_mean_interval

RESULTS_DIR = repo_root / 'assets' / 'evaluation'
REVIEW_DIR = RESULTS_DIR / 'human_review'
BUNDLE_PATH = REVIEW_DIR / 'blinded_review_trajectories.npz'
MASTER_KEY_PATH = REVIEW_DIR / 'blinded_review_key.json'
if not BUNDLE_PATH.is_file() or not MASTER_KEY_PATH.is_file():
    raise FileNotFoundError('Run Notebook 05 completely to prepare the blinded review materials.')

PARTICIPANT_ID = 'reviewer_01'
if not re.fullmatch(r'[A-Za-z0-9_-]+', PARTICIPANT_ID):
    raise ValueError('PARTICIPANT_ID may contain only letters, numbers, underscores, and hyphens.')
participant_dir = REVIEW_DIR / PARTICIPANT_ID
participant_dir.mkdir(parents=True, exist_ok=True)
PARTICIPANT_KEY_PATH = participant_dir / 'method_key.json'
RATINGS_TEMPLATE_PATH = participant_dir / 'ratings_template.csv'
if any(path.exists() for path in (PARTICIPANT_KEY_PATH, RATINGS_TEMPLATE_PATH)):
    raise FileExistsError(f'Review materials already exist for {PARTICIPANT_ID}; choose a new PARTICIPANT_ID to avoid overwriting ratings.')

bundle = np.load(BUNDLE_PATH)
base_trajectories = bundle['trajectories']
pair_positions = bundle['pair_positions'].astype(int)
time_indices = bundle['time_indices'].astype(int)
master_key = json.loads(MASTER_KEY_PATH.read_text(encoding='utf-8'))
static_timepoints = int(master_key['timepoints'])
if len(base_trajectories) != len(pair_positions):
    raise ValueError('Review bundle and pair-position metadata do not match.')

assignment_seed = secrets.randbits(32)
rng = np.random.default_rng(assignment_seed)
blind_trajectories = np.empty_like(base_trajectories)
participant_key = {
    'participant_id': PARTICIPANT_ID,
    'assignment_seed': assignment_seed,
    'time_indices': time_indices.tolist(),
    'tasks': {},
}
for pair_index in range(len(base_trajectories)):
    pair_id = f'pair_{pair_index:03d}'
    original_mapping = master_key['tasks'][pair_id]
    methods = rng.permutation(list(original_mapping.values())).tolist()
    new_mapping = {}
    for code_index, method in enumerate(methods):
        new_code = chr(ord('A') + code_index)
        old_code = next(code for code, name in original_mapping.items() if name == method)
        old_index = ord(old_code) - ord('A')
        blind_trajectories[pair_index, code_index] = base_trajectories[pair_index, old_index]
        new_mapping[new_code] = method
    participant_key['tasks'][pair_id] = new_mapping

PARTICIPANT_KEY_PATH.write_text(json.dumps(participant_key, indent=2) + '\n', encoding='utf-8')
with RATINGS_TEMPLATE_PATH.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=['rater_id', 'pair_id', 'blind_code', 'plausibility_1_5', 'progression_1_5', 'abruptness_1_5', 'comment'],
    )
    writer.writeheader()
    for pair_index in range(len(blind_trajectories)):
        pair_id = f'pair_{pair_index:03d}'
        for code_index in range(blind_trajectories.shape[1]):
            writer.writerow({
                'rater_id': PARTICIPANT_ID,
                'pair_id': pair_id,
                'blind_code': chr(ord('A') + code_index),
                'plausibility_1_5': '',
                'progression_1_5': '',
                'abruptness_1_5': '',
                'comment': '',
            })
print(f'Prepared {len(blind_trajectories)} blinded trajectory assignments.')
print(f'Rating template (share with rater): {RATINGS_TEMPLATE_PATH}')
print(f'Keep this key private until ratings are locked: {PARTICIPANT_KEY_PATH}')

## Rating Instructions

Compare the five coded trajectories for each pair. The static sheet shows 11 frames, while the GIF shows all 51 integration steps side by side. Each row or animation panel represents one method, identified by a random code. The source and target classes are Sneaker and Ankle Boot. The target image is not shown.

Rate each trajectory independently on a scale of 1–5:

- **Plausibility:** 1 = clearly unrealistic, 5 = visually realistic throughout.
- **Progression:** 1 = no clear movement toward an Ankle Boot, 5 = clear and consistent progress toward an Ankle Boot.
- **Abruptness:** 1 = smooth changes, 5 = strong or disruptive jumps. Lower is better.

Use the same scale for all trajectories. Do not try to identify the methods or discuss their identities. You may add a short note about visible artifacts. Do not include personal or sensitive information.

Share only the generated `reviewer_package.zip`, never the private method key. If possible, have multiple people rate the trajectories independently. Do not collect names or other personal data.

For formal or publishable research involving human participants, follow the applicable consent and ethics-review requirements. This is a small pilot study, so its results should not be treated as representative of a larger population.

Clarify the rating workflowAdd an explicit rating template

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter

review_images = blind_trajectories
sampled_frame_indices = np.linspace(0, len(time_indices) - 1, static_timepoints, dtype=int)
review_times = time_indices / max(time_indices)
for pair_index, pair_images in enumerate(review_images):
    fig, axes = plt.subplots(5, len(sampled_frame_indices), figsize=(16, 6.5))
    for method_index in range(5):
        for plot_index, frame_index in enumerate(sampled_frame_indices):
            axis = axes[method_index, plot_index]
            image = pair_images[method_index, frame_index].squeeze()
            axis.imshow(np.clip((image + 1.0) / 2.0, 0.0, 1.0), cmap='gray', vmin=0, vmax=1)
            axis.set_xticks([])
            axis.set_yticks([])
            if method_index == 0:
                axis.set_title(f't={review_times[frame_index]:.1f}', fontsize=8)
        axes[method_index, 0].set_ylabel(f'Code {chr(ord("A") + method_index)}', rotation=0, labelpad=28, va='center')
    fig.suptitle(f'Pair {pair_index:03d}: Sneaker to Ankle Boot', y=1.01)
    fig.tight_layout()
    fig.savefig(participant_dir / f'pair_{pair_index:03d}.png', dpi=150, bbox_inches='tight')
    display(fig)
    plt.close(fig)

    animation_fig, animation_axes = plt.subplots(1, 5, figsize=(10, 2.5))
    image_artists = []
    for method_index, axis in enumerate(animation_axes):
        first_image = pair_images[method_index, 0].squeeze()
        artist = axis.imshow(np.clip((first_image + 1.0) / 2.0, 0.0, 1.0), cmap='gray', vmin=0, vmax=1, animated=True)
        axis.set_title(f'Code {chr(ord("A") + method_index)}')
        axis.set_xticks([])
        axis.set_yticks([])
        image_artists.append(artist)

    def update(frame_index):
        for method_index, artist in enumerate(image_artists):
            image = pair_images[method_index, frame_index].squeeze()
            artist.set_data(np.clip((image + 1.0) / 2.0, 0.0, 1.0))
        animation_fig.suptitle(f'Pair {pair_index:03d} at t={review_times[frame_index]:.2f}')
        return image_artists

    animation = FuncAnimation(
        animation_fig,
        update,
        frames=len(time_indices),
        interval=125,
        blit=True,
    )
    animation.save(participant_dir / f'pair_{pair_index:03d}.gif', writer=PillowWriter(fps=8))
    plt.close(animation_fig)
package_path = participant_dir / 'reviewer_package.zip'
with ZipFile(package_path, 'w', compression=ZIP_DEFLATED) as package:
    package.write(RATINGS_TEMPLATE_PATH, arcname=RATINGS_TEMPLATE_PATH.name)
    for review_file in sorted(participant_dir.glob('pair_*')):
        if review_file.suffix in {'.png', '.gif'}:
            package.write(review_file, arcname=review_file.name)
print(f'Static review sheets and full-step animations saved in {participant_dir}')
print(f'Share only this package; it excludes the private method key: {package_path}')

## Analyze locked ratings

Only run the next cell after all ratings are complete and the completed CSV files and their corresponding private key files are assembled. Do not share the key with raters!
For multiple raters, create one blinded assignment and key per rater. Add each ratings/key pair to ANALYSIS_INPUTS below. The bootstrap resamples reviewed image pairs, so its intervals describe this pilot cohort conditional on the recruited raters.

In [ ]:
ANALYSIS_INPUTS = [
    (participant_dir / 'ratings_completed.csv', PARTICIPANT_KEY_PATH),
]

required_columns = {
    'rater_id', 'pair_id', 'blind_code', 'plausibility_1_5',
    'progression_1_5', 'abruptness_1_5',
}
metric_columns = ('plausibility_1_5', 'progression_1_5', 'abruptness_1_5')
records = []
seen_ratings = set()
missing_inputs = [str(ratings) for ratings, key in ANALYSIS_INPUTS if not ratings.is_file() or not key.is_file()]
if missing_inputs:
    print('Human-rating analysis not run yet. Add completed rating CSVs and matching key files:')
    for path in missing_inputs:
        print(f' - {path}')
else:
    for ratings_path, key_path in ANALYSIS_INPUTS:
        key = json.loads(key_path.read_text(encoding='utf-8'))
        with ratings_path.open(newline='', encoding='utf-8') as handle:
            reader = csv.DictReader(handle)
            if not required_columns.issubset(reader.fieldnames or []):
                raise ValueError(f'Rating file is missing required columns: {ratings_path}')
            for row in reader:
                if not any((row.get(column) or '').strip() for column in metric_columns):
                    continue
                rater_id, pair_id, code = row['rater_id'].strip(), row['pair_id'].strip(), row['blind_code'].strip()
                if pair_id not in key['tasks'] or code not in key['tasks'][pair_id]:
                    raise ValueError(f'Unknown pair/code {pair_id}/{code} in {ratings_path}')
                if not rater_id:
                    raise ValueError(f'Missing rater_id for {pair_id}/{code} in {ratings_path}')
                identity = (rater_id, pair_id, code)
                if identity in seen_ratings:
                    raise ValueError(f'Duplicate rating row for {identity}')
                seen_ratings.add(identity)
                entry = {'rater_id': rater_id, 'pair_id': pair_id, 'method': key['tasks'][pair_id][code]}
                for column in metric_columns:
                    value = (row.get(column) or '').strip()
                    if not value:
                        raise ValueError(f'Missing {column} for {identity}')
                    score = int(value)
                    if score < 1 or score > 5:
                        raise ValueError(f'{column} must be an integer from 1 to 5; found {value!r}')
                    entry[column] = score
                records.append(entry)

if records:
    methods = ('rectified_flow', 'identity', 'global_shift', 'linear_oracle', 'nn_linear_oracle')
    metrics = {
        'plausibility_1_5': 'Plausibility',
        'progression_1_5': 'Target progression',
        'abruptness_1_5': 'Abruptness (lower is better)',
    }
    by_pair_method = {}
    for record in records:
        for metric in metrics:
            by_pair_method.setdefault((record['pair_id'], record['method'], metric), []).append(record[metric])
    pair_scores = {key: float(np.mean(values)) for key, values in by_pair_method.items()}

    agreement_rows = []
    for metric, label in metrics.items():
        repeated = [values for (pair_id, method, item_metric), values in by_pair_method.items()
                    if item_metric == metric and len(values) > 1]
        if repeated:
            exact_agreement = np.mean([np.unique(values, return_counts=True)[1].max() / len(values) for values in repeated])
            within_task_sd = np.mean([np.std(values) for values in repeated])
            agreement_rows.append((label, exact_agreement, within_task_sd, len(repeated)))

    summary_rows = []
    paired_rows = []
    for metric, label in metrics.items():
        for method in methods:
            values = [score for (pair_id, item_method, item_metric), score in pair_scores.items()
                      if item_method == method and item_metric == metric]
            if values:
                mean, lower, upper = bootstrap_mean_interval(values, seed=2031)
                summary_rows.append((label, method, mean, lower, upper, len(values)))
        rf_scores = {
            pair_id: score for (pair_id, method, item_metric), score in pair_scores.items()
            if method == 'rectified_flow' and item_metric == metric
        }
        for method in methods[1:]:
            baseline_scores = {
                pair_id: score for (pair_id, item_method, item_metric), score in pair_scores.items()
                if item_method == method and item_metric == metric
            }
            paired_ids = sorted(rf_scores.keys() & baseline_scores.keys())
            if paired_ids:
                differences = np.asarray([baseline_scores[pair_id] - rf_scores[pair_id] for pair_id in paired_ids])
                mean, lower, upper = bootstrap_mean_interval(differences, seed=2032)
                paired_rows.append((label, method, mean, lower, upper, len(paired_ids)))

    summary_table = '| Criterion | Method | Mean rating [bootstrap 95% interval] | Pairs |\n|---|---|---:|---:|\n'
    for label, method, mean, lower, upper, count in summary_rows:
        summary_table += f'| {label} | {method} | {mean:.2f} [{lower:.2f}, {upper:.2f}] | {count} |\n'
    display(Markdown(summary_table))

    distribution_table = '| Criterion | Method | Rating counts (1, 2, 3, 4, 5) |\n|---|---|---:|\n'
    for metric, label in metrics.items():
        for method in methods:
            scores = [record[metric] for record in records if record['method'] == method]
            if scores:
                counts = np.bincount(scores, minlength=6)[1:6].tolist()
                distribution_table += f"| {label} | {method} | {counts} |\n"
    display(Markdown(distribution_table))

    if agreement_rows:
        agreement_table = '| Criterion | Exact agreement | Mean within-task SD | Tasks with repeated ratings |\n|---|---:|---:|---:|\n'
        for label, exact, spread, count in agreement_rows:
            agreement_table += f'| {label} | {exact:.1%} | {spread:.2f} | {count} |\n'
        display(Markdown(agreement_table))
    else:
        print('Only one rating per trajectory was available; inter-rater agreement cannot be estimated.')

    paired_table = '| Criterion | Baseline | Baseline − RF [bootstrap 95% interval] | Pairs |\n|---|---|---:|---:|\n'
    for label, method, mean, lower, upper, count in paired_rows:
        paired_table += f'| {label} | {method} | {mean:.2f} [{lower:.2f}, {upper:.2f}] | {count} |\n'
    display(Markdown(
        paired_table + '\nPositive differences favor the baseline for plausibility/progression; negative differences favor it for abruptness.'
    ))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for axis, (metric, title) in zip(axes, metrics.items()):
        rows = [row for row in summary_rows if row[0] == title]
        labels = [row[1].replace('_', ' ') for row in rows]
        means = [row[2] for row in rows]
        errors = [[row[2] - row[3] for row in rows], [row[4] - row[2] for row in rows]]
        axis.bar(labels, means, yerr=errors, capsize=3)
        axis.set(title=title, ylabel='Mean rating (1–5)', ylim=(1, 5))
        axis.tick_params(axis='x', labelrotation=45)
        axis.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    plt.show()
    print('Ratings are ordinal; means and exact agreement are descriptive. Intervals resample image pairs and do not establish broad human preference or population-level significance.')
else:
    print('No complete human ratings were loaded. Complete the rating template, set the ratings_completed.csv path, and rerun this cell.')